# 20 — Feature Selection Scenarios

The pipeline compares **four input scenarios** for segmentation:

| key | scenario | description |
|-----|----------|-------------|
| `single_date` | Single-date baseline | peak-NDVI date, all 10 bands |
| `mt_base` | Multi-temporal NDVI baseline | 4 calendar dates × VEGE_BANDS |
| `gsi` | **GSI-direct** | per-crop Global Separability Index, top-K union |
| `rf`  | **RF-direct** | per-crop Random-Forest importance, top-K union |

This notebook runs the two *selection* methods (GSI, RF) and inspects their
chosen channel sets. Driven by `stages/band_scoring.py`.

> Scoring reads the full processed S2 stack and is heavy (GPU for GSI CNN scoring,
> CPU-bound RF). Cells that recompute are guarded by existing outputs; delete the
> output JSON or pass `force=True` to re-run.

In [ ]:
# Make the pipeline importable as `crop_mapping_pipeline` regardless of the
# checkout directory name (this repo is `cropmap-remote-sensing-exps`; the
# GPU deploy dir is `crop_mapping_pipeline`). Also silence MLflow telemetry.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'crop_mapping_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from crop_mapping_pipeline import config as C
print('Repo :', REPO)
print('Classes:', C.NUM_CLASSES, '| crops:', list(C.CDL_CLASS_NAMES.values()))
print('S2 bands/date:', C.S2_BAND_NAMES)
print('S2 train dir :', C.S2_TRAIN_DIR)
print('CDL train    :', C.CDL_TRAIN)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from crop_mapping_pipeline.stages import band_scoring

## 1. GSI scoring (Stage-1 candidates)

Computes per-crop Global Separability Index over all (date × band) channels →
`gsi_candidates.json`.

In [ ]:
# Heavy: full-stack GSI scoring. Skips if candidates already exist.
band_scoring.main(mode='gsi', force=False)
print('GSI candidates:', C.GSI_CANDIDATES_JSON)

## 2. GSI-direct selection (top-K union)

Ranks channels by per-crop SI_global, keeps top-K per crop, unions across crops.

In [ ]:
K = C.SELECT_TOP_K_PER_CROP
band_scoring.main(mode='select', selector='gsi_direct', top_k_values=[K], force=False)
print('GSI-direct →', C.SELECT_GSI_DIRECT_JSON)

## 3. RF-direct selection (top-K union)

Multi-class Random Forest (MDI importance) over all channels, top-K per crop, union.

In [ ]:
band_scoring.main(mode='select', selector='rf_direct', top_k_values=[K], force=False)
print('RF-direct →', C.SELECT_RF_DIRECT_JSON)

## 4. Inspect + compare the selected channel sets

GSI keeps a broad, year-spread set; RF concentrates on far fewer channels.

In [ ]:
def load_channels(json_path, k=K):
    p = Path(str(json_path).replace('.json', f'_k{k}.json'))
    if not p.exists(): p = Path(json_path)
    if not p.exists():
        print('missing:', p); return None, None
    d = json.loads(p.read_text())
    # union of channel names/indices across crops (schema-tolerant)
    chans = set()
    for v in (d.values() if isinstance(d, dict) else []):
        if isinstance(v, dict):
            for key in ('channels', 'bands', 'selected', 'indices'):
                if key in v: chans.update(map(str, v[key]))
        elif isinstance(v, list):
            chans.update(map(str, v))
    return d, sorted(chans)

gsi_d, gsi_ch = load_channels(C.SELECT_GSI_DIRECT_JSON)
rf_d,  rf_ch  = load_channels(C.SELECT_RF_DIRECT_JSON)
if gsi_ch is not None and rf_ch is not None:
    gset, rset = set(gsi_ch), set(rf_ch)
    print(f'GSI-direct: {len(gset)} channels')
    print(f'RF-direct : {len(rset)} channels')
    print(f'Shared    : {len(gset & rset)} | GSI-only {len(gset-rset)} | RF-only {len(rset-gset)}')
    plt.figure(figsize=(5, 4))
    plt.bar(['GSI-direct', 'RF-direct'], [len(gset), len(rset)],
            color=['steelblue', 'darkorange'])
    plt.ylabel('# channels selected'); plt.title(f'Selected channel count (top-K={K})')
    plt.tight_layout(); plt.show()